In [13]:
import json
from IPython.display import Markdown, display
import time
from elasticsearch import Elasticsearch, helpers

In [22]:
# refine the corpus
with open("corpus_fau.eu.jsonl", "r", encoding="utf-8") as f, open("cleaned_corpus_fau.eu.jsonl", "w", encoding="utf-8") as out:
    for doc in f:
        doc = json.loads(doc)
        split = doc["text"].split("---\n")
        metadata = split[1].strip()
        text = "---\n".join(split[2:]).strip()
        metadict = dict()
        for m in metadata.split("\n"):
            key, val = m.split(":")[0].strip(), ":".join(m.split(":")[1:]).strip()
            metadict[key] = val
        doc.update({"text": text, "metadata": metadict})
        out.write(json.dumps(doc, ensure_ascii=False)+"\n")


In [6]:
import json
corpus = list()
# with open("main_corpus_nhr_fau_de_eu.jsonl", "r", encoding="utf-8") as f:
with open("crawl4ai_corpus.jsonl", "r", encoding="utf-8") as f:
    for doc in f:
        doc = json.loads(doc)
        corpus.append(doc)

In [7]:
len(corpus)

6799

In [14]:
client = Elasticsearch(
    # For local development
    "http://localhost:9200",
    basic_auth=("elastic", "zb3BaJvO")
)

In [15]:
client.cluster.health()

ObjectApiResponse({'cluster_name': 'docker-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 50, 'active_shards': 50, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 4, 'unassigned_primary_shards': 0, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 92.5925925925926})

In [14]:
batch_size = 2000
x = [(i*batch_size, min(len(corpus), (i+1)*batch_size)) for i in range(len(corpus)//batch_size+1)]
[(i, j) for i,j in x]

[(0, 2000), (2000, 4000), (4000, 6000), (6000, 7064)]

In [8]:
# adidng data to index
# index "search-test02" is trafilatura pages
# index "search-cai" is crawl4ai files
batch_size = 500
batch_indices = [(i*batch_size, min(len(corpus), (i+1)*batch_size)) for i in range(len(corpus)//batch_size+1)]  # [(0, 2000), (2000, 4000), (4000, 6000), (6000, 7064)]
batches = [corpus[i:j] for i,j in batch_indices]
for batch in batches:
    operations = list()
    for doc in batch:
        try:
            doc = {
                    "text": doc["text"],
                    "url": doc["url"],
                    "title": doc["metadata"].get("title", ""),
                    "description": doc["metadata"].get("description", ""),
                }
            operations.append({"_index": "search-cai", "_source": doc})
        except KeyError as e:
            pass
    print(f"bulk opperation: on {len(batch)} documents")
    res = helpers.bulk(client, operations)
    time.sleep(1)
"done"

bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 500 documents
bulk opperation: on 299 documents


'done'

In [31]:
help(client.search)

Help on method search in module elasticsearch:

search(*, index: Union[str, Sequence[str], NoneType] = None, aggregations: Optional[Mapping[str, Mapping[str, Any]]] = None, aggs: Optional[Mapping[str, Mapping[str, Any]]] = None, allow_no_indices: Optional[bool] = None, allow_partial_search_results: Optional[bool] = None, analyze_wildcard: Optional[bool] = None, analyzer: Optional[str] = None, batched_reduce_size: Optional[int] = None, ccs_minimize_roundtrips: Optional[bool] = None, collapse: Optional[Mapping[str, Any]] = None, default_operator: Union[str, Literal['and', 'or'], NoneType] = None, df: Optional[str] = None, docvalue_fields: Optional[Sequence[Mapping[str, Any]]] = None, error_trace: Optional[bool] = None, expand_wildcards: Union[Sequence[Union[str, Literal['all', 'closed', 'hidden', 'none', 'open']]], str, Literal['all', 'closed', 'hidden', 'none', 'open'], NoneType] = None, explain: Optional[bool] = None, ext: Optional[Mapping[str, Any]] = None, fields: Optional[Sequence[M

In [12]:
res = client.search(
    index="search-cai",
    explain=True,
    size=20,
    query={
        # "match": {
        #     # "text": "examination period summer semester 2025"
        #     # "text": "German universities for Materials Science Master's according to CHE"
        #     "text": "NHR cluster overview",
        # }
        "multi_match" : {
            # "query":    "NHR HPC cluster overview",
            "query":    "FAU Bachelorstudiengänge Sozialwissenschaften",
            "fields": ["text", "title", "description", "url"]
        }
    }
)
with open("hits.json", "w") as f:
    json.dump(res.body, f, indent = 4)
for hit in res.body["hits"]["hits"]:
    # display(Markdown(hit["_source"]["url"]))
    display(Markdown(20*"🔶\n"+hit["_source"]["url"]+"\n"+hit["_source"]["text"]))
    # display(40*"🔶")
# print(json.dumps(res.body["hits"]["hits"], indent=4))

🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Fachbereich Wirtschafts- und Sozialwissenschaften
# Fachbereich Wirtschafts- und Sozialwissenschaften
Rahmen- und Fachprüfungsordnungen der Studiengänge am Fachbereich Wirtschafts- und Sozialwissenschaften
### Rahmenprüfungsordnung
##  Rahmenprüfungsordnung für Bachelorstudiengänge 
konsolidierte Fassungen | Dateigröße  
---|---  
[Bachelorstudiengänge BPOWISO 20240807.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Bachelor/Rahmenpruefungsordnung/konsolidierte_Fassungen/Bachelorstudieng%C3%A4nge_BPOWISO_20240807.pdf) | 275 KB  
[Bachelorstudiengänge BPOWISO 20060801 i.d.F. 20230323.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Bachelor/Rahmenpruefungsordnung/konsolidierte_Fassungen/Bachelorstudiengaenge_BPOWISO_20060801_idF_20230323.pdf) | 368 KB  
[Bachelorstudiengänge BPOWISO 20060801 i.d.F. 20200902.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Bachelor/Rahmenpruefungsordnung/konsolidierte_Fassungen/Bachelorstudiengaenge_BPOWISO_20060801_idF_20200902.pdf) | 571 KB  
[Bachelorstudiengänge BPOWISO 20060801 i.d.F. 20190614.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Bachelor/Rahmenpruefungsordnung/konsolidierte_Fassungen/Bachelorstudiengaenge_BPOWISO_20060801_idF_20190614.pdf) | 569 KB  
Änderungssatzungen | Dateigröße  
---|---  
[BPOWISO 20230323 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Bachelor/Rahmenpruefungsordnung/Aenderungssatzungen/BPOWISO_20230323_AeS.pdf) | 436 KB  
[BPOWISO 20200902 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Bachelor/Rahmenpruefungsordnung/Aenderungssatzungen/BPOWISO_20200902_AeS.pdf) | 230 KB  
[BPOWiWi 20190614 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Bachelor/Rahmenpruefungsordnung/Aenderungssatzungen/BPOWiWi_20190614_AeS.pdf) | 552 KB  
**Änderungssatzung/Datum** | **Konsolidierte Fassung/Datum** | **English version as amended on**  
---|---|---  
[10. August 2017](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/18AeSa_BPOWiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 10.08.2017](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_BPOWiWi_AUG2017.pdf)) |   
[15. Juli 2016](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/17AES_BA-WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 15.07.2016](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_JULI2016.pdf)) |   
[29. Februar 2016](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/16AES_BA-WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 29.02.2016](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_FEB2016.pdf)) |   
[23. Juli 2015](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/15AES_BA-WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 23.07.2015](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_JULI2015.pdf)) |   
[25. Juli 2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/14AES_BA-WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 25.07.2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_JULI2014.pdf)) |   
[10. Januar 2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/13AES_BA_WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 10.01.2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_JAN2014.pdf)) |   
[26. Juli 2013](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/12AES_BA_WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 26.07.2013](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_JULI2013.pdf)) |   
[13. Februar 2013](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/11AES%20BA-WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 13.02.2013](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_FEB2013.pdf)) |   
[1. August 2012 ](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/10AES-BA%20WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 01.08.2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_AUG2012.pdf)) |   
[24. Februar 2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/9AES-BA%20WiWi.pdf) | ([PDF vom 01.08.2006 i.d.F. 24.02.2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_24FEB2012.pdf)) |   
[24. Februar 2011](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/8AeS-BA%20WiWi_1.pdf) | ([PDF vom 01.08.2006 i.d.F. 24.02.2011](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_24FEB2011.pdf)) |   
[30. Juli 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/7AES-BA%20WiWi.pdf) ab WiSe 2010/11 | ([PDF vom 01.08.2006 i.d.F. 30.07.2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_NEU-WS2010-2011.pdf)) |   
[24. Februar 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/6AES-BA%20WiWi.pdf) bis SoSe 2010 | ([PDF vom 01.08.2006 i.d.F. 24.02.2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO_Bachelorstudiengaenge_NEU.pdf)) |   
[28. August 2009](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/5AeSa-BA_WiWi.pdf) |  |   
[19. März 2009](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/4AeSa-BA_WiWi.pdf) |  |   
[28. Februar 2008](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/3AeSa-BA_WiWi.pdf) |  |   
[9. Oktober 2007](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/2AeSa-BA_WiWi.pdf) |  |   
[26. Juni 2007](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/1AeSa-BA_WiWi.pdf) |  |   
| ([PDF vom 01.08.2006](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/ReWi-Urfassung/FB%20WiWi/PO%20-%20BA%20-%20WiWi/PrO-BPO%20WiSo%20mit%20Anlagen.pdf)) |   
##  Rahmenprüfungsordnung für Masterstudiengänge 
konsolidierte Fassungen | Dateigröße  
---|---  
[Rahmenprüfungsordnung MPOWISO 2024807.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Master/Rahmenpruefungsordnung/konsolidierte_Fassungen/Rahmenpr%C3%BCfungsordnung_MPOWISO_2024807.pdf) | 308 KB  
[Rahmenprüfungsordnung MPOWISO 20090716 i.d.F. 20230731.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Master/Rahmenpruefungsordnung/konsolidierte_Fassungen/Rahmenpr%C3%BCfungsordnung_MPOWISO_20090716_idF_20230731.pdf) | 288 KB  
[Rahmenprüfungsordnung MPOWISO 20090716 i.d.F. 20191120.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Master/Rahmenpruefungsordnung/konsolidierte_Fassungen/Rahmenpr%C3%BCfungsordnung_MPOWISO_20090716_idF_20191120.pdf) | 527 KB  
Änderungssatzungen | Dateigröße  
---|---  
[Sammel-ÄSa Masterbewerbung 20230731 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Master/Rahmenpruefungsordnung/Aenderungssatzungen/Sammel-AeSa_Masterbewerbung_20230731_AeS.pdf) | 152 KB  
[Rahmenprüfungsordnung MPOWiWi 20201019 ÄS zu 11ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Master/Rahmenpruefungsordnung/Aenderungssatzungen/Rahmenpr%C3%BCfungsordnung_MPOWiWi_20201019_AeS_zu_11AeS.pdf) | 205 KB  
[Rahmenprüfungsordnung MPOWIWI 20191120 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/RW/WiWi/Master/Rahmenpruefungsordnung/Aenderungssatzungen/Rahmenpr%C3%BCfungsordnung_MPOWIWI_20191120_AeS.pdf) | 249 KB  
**Änderungssatzung/Datum** | **Konsolidierte Fassung/Datum** | **English version as amended on**  
---|---|---  
[18. August 2017](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/10AeSa_MPOWiWi.pdf) | ([PDF vom 16.07.2009 i.d.F. 18.08.2017](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO-MA-WiWi-AUG2017.pdf)) |   
[25. Juli 2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/9AES_RPO-MA-WiWi.pdf) | ([PDF vom 16.07.2009 i.d.F. 25.07.2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO-MA-WiWi-JULI2014.pdf)) |   
[26. Juli 2013](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/8AES_RPO-MA-WiWi.pdf) | ([PDF vom 16.07.2009 i.d.F. 26.07.2013](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO-MA-WiWi-JULI2013.pdf)) |   
[13. Februar 2013](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/7AES%20RPO-MA-WiWi.pdf) | ([PDF vom 16.07.2009 i.d.F. 13.02.2013](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO-MA-WiWi-FEB2013.pdf)) |   
[1. August 2012 ](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/6AES%20RPO-MA%20WiWi.pdf) | ([PDF vom 16.07.2009 i.d.F. 01.08.2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO-MA-WiWi-AUGUST2012.pdf)) |   
[19. Januar 2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/5AES%20RPO-MA%20WiWi.pdf) | ([PDF vom 16.07.2009 i.d.F. 19.01.2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO-MA-WiWi-JANUAR2012.pdf)) |   
[3. März 2011](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/4AES%20RPO-MA%20WiWi.pdf) | ([PDF vom 16.07.2009 i.d.F. 03.03.2011](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO-MA-WiWi-MAERZ2011.pdf)) |   
[30. Juli 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/3AES%20RPO-MA%20WiWi.pdf) | ([PDF vom 16.07.2009 i.d.F. 30.07.2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/PO-MA-WiWi.pdf)) |   
[24. Februar 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/2AES%20RPO-MA%20WiWi.pdf) |  |   
[18. Januar 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/Aenderungssatzungen/1AES%20RPO-MA%20WiWi.pdf) |  |   
| ([PDF vom 16.07.2009](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/WISO/ReWi-Urfassung/FB%20WiWi/RPO-MA%20WiWi.pdf)) |   
### Fachprüfungsordnungen
Die folgenden Webseiten der Prüfungsordnungen haben teilweise sehr lange Ladezeiten!
##  Bachelorstudiengänge 
  * [International Business Studies](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/bachelor/#mibs)
  * [International Economic Studies](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/bachelor/#mies)
  * [Sozialökonomik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/bachelor/#sozialoekonomik)
  * [Wirtschaftsinformatik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/bachelor/#wirtschaftsinformatik)
  * [Wirtschaftswissenschaften](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/bachelor/#wirtschaftswissenschaften)


##  Masterstudiengänge 
  * [Arbeitsmarkt und Personal](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#arbeitsmarkt-personal)
  * [Economics](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#economics)
  * [Finance, Auditing, Controlling, Taxation](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#finance-auditing-controlling-taxation)
  * [Gesundheitsmanagement und Gesundheitsökonomie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#gesundheitsmanagement)
  * [International Business Studies](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#ibs)
  * [Internationale Wirtschaftsinformatik/International Information Systems](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#iis)
  * [Management](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#management)
  * [Marketing](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#marketing)
  * [Sozialökonomik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#sozialoekonomik-ma)
  * [Wirtschaftspädagogik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/#wirtschaftspaedagogik-ma)


##  Weiterbildungsmaster (berufsbegleitend) 
  * [Business Management (MBA)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/weiterbildungsmaster/#business-management)
  * [Digital Business (MDBA) (ab WS 2024/25 Digital Business & AI)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/weiterbildungsmaster/#digital-business)
  * [Digital Business & AI (MBA) (bis WS 2024/25 Digital Business)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/weiterbildungsmaster/#digital-business-and-AI)
  * [Global Business Management (MBA)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/weiterbildungsmaster/#global-business-management)
  * [Health Business Administration (MHBA)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/weiterbildungsmaster/#mhba)
  * [Marketing- und Vertriebsmanagement (M.Sc.)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/weiterbildungsmaster/#mmm)
  * [Sustainability Management (MBA)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/weiterbildungsmaster/#sustainability-management)


##  Modul- und Zusatzstudien 
  * [Modulstudien Berufspädagogik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/modul-und-zusatzstudien/)


### Weitere Regelungen
  * [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/sprachpruefungen/ "Sprachprüfungen")
  * [Promotions-und Habilitationsordnungen](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions-und Habilitationsordnungen")
  * [Diplomstudiengänge und Aufbaustudiengänge am Fachbereich Wirtschaftswissenschaften](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rechts-und-wirtschaftswissenschaftliche-fakultaet/wiso/diplomstudiengaenge-und-weiteres/ "Diplomstudiengänge und Aufbaustudiengänge am Fachbereich Wirtschaftswissenschaften")




🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/termine/digitaler-bachelorday-studieninfos-an-der-fau-wiso
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Digitaler BachelorDay: Studieninfos an der FAU WiSo
# Digitaler BachelorDay: Studieninfos an der FAU WiSo
Datum: 1. Juli 2025Zeit: 10:00 – 15:30Ort: Zoom
Welche Bachelorstudiengänge gibt es an der FAU WiSo? Wie sieht das Studierendenleben in Nürnberg aus? Antworten auf diese und weitere Fragen gibt es beim digitalen BachelorDay des Fachbereichs Wirtschafts- und Sozialwissenschaften (WiSo) der Friedrich-Alexander-Universität Erlangen-Nürnberg (FAU) am Dienstag, den 01. Juli 2025.  
Der BachelorDay der Nürnberger WiSo ist die perfekte Möglichkeit, sich von überall auf der Welt via Computer, Tablet oder Smartphone über das Studienangebot und das Campusleben am Fachbereich zu informieren. Die Referentinnen und Referenten beantworten Interessierten einen Tag lang Fragen rund um das Bachelorstudium an der FAU WiSo.  
Es stellen sich die Bachelorstudiengänge Wirtschaftswissenschaften, Sozialökonomik, Wirtschaftsingenieurwesen, International Production Engineering and Management und Wirtschaftsinformatik sowie die zwei internationalen Bachelor International Business Studies und International Economic Studies, welche komplett in englischer Sprache unterrichtet werden, in kompakten, 45-minütigen Online-Vorträgen vor.
Des Weiteren gibt es allgemeine Informationen über das Bewerbungs- und Zulassungsverfahren von der Zentralen Studienberatung der FAU. Wer möchte, kann beim virtuellen Campusrundgang die wichtigsten Anlaufstellen und Einrichtungen besuchen und einen Blick in die Hörsäle, die Bibliothek und auf den City Campus im Herzen von Nürnberg werfen.
Die Vorträge zu den einzelnen Studiengängen können per Zoom besucht werden. Eine vorherige Registrierung ist nicht notwendig. Die entsprechenden Vortragszeiten, Links und Zugangsdaten finden sich beim jeweiligen Programmpunkt auf der Seite <http://www.wiso.fau.de/bachelorday><<http://www.wiso.fau.de/bachelorday>>.
Weitere Informationen:  
Nina Knauer, nina.knauer@fau.de
[Zum Kalender hinzufügen](https://www.fau.de?ical-plugin=rrze-calendar&action=export&filename=www-fau-de-termine-digitaler-bachelorday-studieninfos-an-der-fau-wiso&ids=18092473)
## Details Datum:
    1. Juli 2025 

Zeit:
    10:00 – 15:30 

Ort:
    
Zoom 

Veranstaltungskategorien:
    [Studieninteressierte](https://www.fau.de/calendars/studieninteressierte/)


🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Studien- und Prüfungsordnungen
# Studien- und Prüfungsordnungen
Studien- und Prüfungsordnungen
## Weitere Informationen
  * [Prüfungsämter](https://www.fau.de/education/beratungs-und-servicestellen/pruefungsaemter/)
  * [Prüfungsverfahren](https://www.intern.fau.de/lehre-und-studium/rechtsangelegenheiten-in-lehre-und-studium/#Pruefungsverfahren)
  * [Regelungen zum Studium](https://www.fau.de/fau/rechtsgrundlagen/regelungen-zum-studium/)
  * [Online-Prüfungsverwaltung „campo“](http://campo.fau.de/)


Hier finden Sie – nach Fakultäten gegliedert – die Studien- und Prüfungsordnungen der an der FAU angebotenen Studiengänge (inklusive Staatsexamen), der sonstigen Studien (zum Beispiel Modulstudien) sowie Regelungen zu Promotions- und Habilitationsverfahren.
[Corona-Satzung der FAU](https://www.fau.de/fau/rechtsgrundlagen/regelungen-zum-studium/#coronasatzung)
[FAU-Satzung über die Durchführung elektronischer Fernprüfungen (EFernPO)](https://www.fau.de/fau/rechtsgrundlagen/regelungen-zum-studium/#EFernPO)
###  Philosophische Fakultät und Fachbereich Theologie 
  1. [Allgemeine Bachelor-/Masterstudien- und Prüfungsordnung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/#apo "Allgemeine Bachelor-/Masterstudien- und Prüfungsordnung")
  2. [Fachstudien- und Prüfungsordnungen: Zwei-Fach-Bachelorstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/ "Fachstudien- und Prüfungsordnungen: Zwei-Fach-Bachelorstudiengänge")
  3. [Fachstudien- und Prüfungsordnungen: Ein-Fach-Bachelorstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/1-fach-bachelor/ "Fachstudien- und Prüfungsordnungen: Ein-Fach-Bachelorstudiengänge")
  4. [Fachstudien- und Prüfungsordnungen: Masterstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/ "Fachstudien- und Prüfungsordnungen: Masterstudiengänge")
  5. [Weiterbildungs- und Elitestudiengänge und weitere](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/)
  6. Fachbereich Theologie ([Fachstudien- und Prüfungsordnungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/philosophische-fakultaet-und-fachbereich-theologie/ "Fachstudien- und Prüfungsordnungen"), [Promotions- und Habilitationsordnung](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions- und Habilitationsordnung"))
  7. [Modulstudien und Zusatzstudien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/modulstudien-und-zusatzstudien/)
  8. [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/sprachpruefungen/ "Sprachprüfungen")
  9. [Promotions- und Habilitationsordnung](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions- und Habilitationsordnung")
  10. [Lehramtsstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/ "Lehramtsstudiengänge")


###  Rechts- und Wirtschaftswissenschaftliche Fakultät 
#### Fachbereich Rechtswissenschaft
  * [Studien- und Prüfungsordnungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/fachbereich-rechtswissenschaft/ "Studien- und Prüfungsordnungen")
  * [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/sprachpruefungen/ "Sprachprüfungen")
  * [Promotions- und Habilitationsordnung](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions- und Habilitationsordnung")
  * [Zusatzstudien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/fachbereich-rechtswissenschaft/#collapse_11)


#### Fachbereich Wirtschafts- und Sozialwissenschaften
  * [Studien- und Prüfungsordnungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/ "Studien- und Prüfungsordnungen")
    * [Rahmenprüfungsordnungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/)
    * [Bachelorstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/bachelor/)
    * [Masterstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/master/)
    * [Weiterbildungsmasterstudiengänge (berufsbegleitend)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/weiterbildungsmaster/)
    * [Diplomstudiengänge und weitere](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/rewi/wiso/diplomstudiengaenge-und-weiteres/)
  * [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/sprachpruefungen/ "Sprachprüfungen")
  * [Promotions- und Habilitationsordnung](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions- und Habilitationsordnung")


###  Medizinische Fakultät 
  * [Studien- und Prüfungsordnungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/med/ "Studien- und Prüfungsordnungen")
  * [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/sprachpruefungen/ "Sprachprüfungen")
  * [Promotions- und Habilitationsordnung](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions- und Habilitationsordnung")


###  Naturwissenschaftliche Fakultät 
  1. [Allgemeine Studien- und Prüfungsordnung für die Bachelor- und Masterstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/nat/#apo)
  2. [Department Biologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/nat/biologie/ "Department Biologie")
  3. [Department Chemie und Pharmazie ](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/nat/chemie-und-pharmazie/)
  4. [Department Geographie und Geowissenschaften](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/nat/geographie/)
  5. [Department Mathematik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/nat/mathematik/)
  6. [Department Physik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/nat/physik/)
  7. [Modulstudien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/nat/weiteres/)
  8. [Spezielle weiterbildende Studien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/nat/weiteres/)
  9. [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Sprachprüfungen")
  10. [Promotions- und Habilitationsordnung](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions- und Habilitationsordnung")
  11. [Lehramtsstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/ "Lehramtsstudiengänge")


###  Technische Fakultät 
  1. [Allgemeine Prüfungsordnungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/tech/#allg-po-ba-ma "Allgemeine Prüfungsordnungen")
  2. [Department Artificial Intelligence in Biomedical Engineering](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/tech/artificial-intelligence-in-biomedical-engineering/#artificial-intelligence-ba)
  3. [Department Informatik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/tech/informatik/)
  4. [Department Elektrotechnik, Elektronik, Informationstechnik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/tech/elektrotechnik-elektronik-informationstechnik/)
  5. [Department Chemie- und Bioingenieurwesen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/tech/chemie-und-bioingenieurwesen/)
  6. [Department Werkstoffwissenschaften](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/tech/werkstoffwissenschaften/)
  7. [Department Maschinenbau](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/tech/maschinenbau/)
  8. [Spezielle weiterbildende Studien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/tech/#weiterbildende-studien)
  9. [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/sprachpruefungen/ "Sprachprüfungen")
  10. [Promotions- und Habilitationsordnung](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions- und Habilitationsordnung")
  11. [Lehramtsstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/ "Lehramtsstudiengänge")


###  Lehramtsstudiengänge 
  * [Erste und zweite Staatsprüfung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/)
  * [Fachstudien- und Prüfungsordnungen der Fächer (FPO)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/lehramtsfaecher/)
  * [LAPO (Modulprüfungen im Rahmen der Ersten Lehramtsprüfung sowie den lehramtsbezogenen Masterstudiengang Gymnasium)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/#module)
  * [Bachelor- und Masterstudiengänge Berufspädagogik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/berufspaedagogik-und-zusatzstudien/)
  * [Zusatzfächer](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/berufspaedagogik-und-zusatzstudien/)


###  Sprachprüfungen 
  * [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/sprachpruefungen/ "Sprachprüfungen")




🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Philosophische Fakultät und Fachbereich Theologie
# Philosophische Fakultät und Fachbereich Theologie
Philosophische Fakultät und Fachbereich Theologie
Es gelten jeweils die Allgemeine Studien- und Prüfungsordnung **und** die Fachstudien- und Prüfungsordnung Ihres Studiengangs!
### Allgemeine Studien- und Prüfungsordnung für die Bachelor- und Masterstudiengänge
##  Allgemeine Studien- und Prüfungsordnung für die Bachelor- und Masterstudiengänge 
konsolidierte Fassungen | Dateigröße  
---|---  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20240807 i.d.F. 20241219.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/konsolidierte_Fassungen/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20240807_idF_20241219.pdf) | 416 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20240807.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/konsolidierte_Fassungen/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20240807.pdf) | 474 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20230822.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/konsolidierte_Fassungen/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20230822.pdf) | 470 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20220629.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/konsolidierte_Fassungen/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20220629.pdf) | 918 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20210806.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/konsolidierte_Fassungen/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20210806.pdf) | 913 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20200806.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/konsolidierte_Fassungen/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20200806.pdf) | 912 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20190828.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/konsolidierte_Fassungen/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20190828.pdf) | 764 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20180801.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/konsolidierte_Fassungen/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20180801.pdf) | 540 KB  
englisch | Dateigröße  
---|---  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20240807 en.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/englisch/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20240807_en.pdf) | 398 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20230822 en.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/englisch/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20070927_idF_20230822_en.pdf) | 442 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20220629 en.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/englisch/Allg_StuO_PrO_BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20220629_en.pdf) | 437 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20210806 en.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/englisch/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20210806_en.pdf) | 435 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20200806 en.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/englisch/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20200806_en.pdf) | 730 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20190828 en.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/englisch/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20190828_en.pdf) | 346 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20070927 i.d.F. 20180801 en.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/englisch/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20070927_idF_20180801_en.pdf) | 701 KB  
Änderungssatzungen | Dateigröße  
---|---  
[Allg StuO Pro BA-MA Phil ABMStPO Phil 20241219 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/Aenderungssatzungen/Allg_StuO_Pro_BA-MA_Phil_ABMStPO_Phil_20241219_AeS.pdf) | 108 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20230822 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/Aenderungssatzungen/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20230822_AeS.pdf) | 112 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20220629 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/Aenderungssatzungen/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20220629_AeS.pdf) | 367 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20210806 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/Aenderungssatzungen/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20210806_AeS.pdf) | 218 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20200806 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/Aenderungssatzungen/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20200806_AeS.pdf) | 365 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20190828 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/Aenderungssatzungen/Allg_StuO_PrO_BA-MA_Phil_ABMStPO_Phil_20190828_AeS.pdf) | 114 KB  
[Allg StuO PrO BA-MA Phil ABMStPO Phil 20180801 ÄS.pdf](https://www.doc.zuv.fau.de//L1/PO/Phil/APO_Bachelor_Master/Aenderungssatzungen/Allg_StuO_PrO_%20BA-MA_%20Phil_ABMStPO_Phil_20180801_AeS.pdf) | 296 KB  
**Änderungssatzung/Datum** | **Konsolidierte Fassung/Datum** | **English version as amended on**  
---|---|---  
[24. August 2017](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/17AeSa_ABMStPO_Phil.pdf) | ([PDF vom 27.09.2007 i.d.F. 24.08.2017](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil_ABMStPO_Phil_AUG2017.pdf)) | ([PDF 24th of August 2017](https://www.zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/pdf-englisch/StuO_PrO_Allg_%20BA_%20Phil_ABMStPO_Phil_AUG2017_EN.pdf))  
[2. August 2016](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/16AES_AllgPrO%20BA-MA%20Phil%20Fak.pdf) | ([PDF vom 27.09.2007 i.d.F. 02.08.2016](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.AUG2016.pdf)) |   
[6. August 2015](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/15AES_AllgPrO%20BA-MA%20Phil%20Fak.pdf) | ([PDF vom 27.09.2007 i.d.F. 06.08.2015](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.AUG2015.pdf)) |   
[21. Juli 2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/14AES_AllgPrO%20BA-MA%20Phil%20Fak.pdf) | ([PDF vom 27.09.2007 i.d.F. 21.07.2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.JULI2014.pdf)) | ([PDF 21st of July 2014)](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/pdf-englisch/StuO_PrO_Allg_%20BA_%20Phil%20JULI2014_en.pdf)  
[19. Februar 2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/13AES_AllgBA-MA%20Phil.pdf) | ([PDF vom 27.09.2007 i.d.F. 19.02.2014](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.FEBRUAR2014.pdf)) |   
[8. Oktober 2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/12AES-Allg%20BA-MA%20Phil.pdf) | ([PDF vom 27.09.2007 i.d.F. 08.10.2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.OKTOBER2012.pdf)) |   
[18. Januar 2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/11AES-Allg%20BA-MA%20Phil.pdf) | ([PDF vom 27.09.2007 i.d.F. 18.01.2012](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.JANUAR2012.pdf)) |   
[5. August 2011](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/10AES-Allg%20BA%20Phil.pdf) | ([PDF vom 27.09.2007 i.d.F. 05.08.2011](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.AUGUST2011.pdf)) |   
[8. März 2011](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/9AES-Allg%20BA%20Phil.pdf) | ([PDF vom 27.09.2007 i.d.F. 08.03.2011](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.MAERZ2011.pdf)) |   
[5. November 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/1AE-1Fach_2Fach_BA-Phil.pdf) | ([PDF vom 27.09.2007 i.d.F. 05.11.2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/StuO_PrO_Allg_%20BA_%20Phil.pdf)) |   
[6. Juli 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/7AES-Allg%20BA%20Phil.pdf) |  |   
[1. Juni 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/6AES-Allg%20BA%20Phil.pdf) |  |   
[3. März 2010](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/5AES-Allg%20BA%20Phil_1.pdf) |  |   
[4. September 2009](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/4AES-Allg%20BA%20Phil.pdf) |  |   
[1. September 2009](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/3AeSa-Allg%20BA%20Phil.pdf) |  |   
[5. August 2008](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/2AeSa_ABStPO_Phil.pdf) |  |   
[3. Dezember 2007](https://zuv.fau.de/universitaet/organisation/recht/studiensatzungen/PHIL1/Aenderungssatzungen/1_AeSa_AllgPrO_Phil.pdf) |  |   
| ([PDF vom 27.09.2007](https://zuv.fau.de/universitaet/organisation/recht/FW-Urfassungen/StuOPrOAllgBAPhil.pdf)) |   
### Fachstudien- und Prüfungsordnungen
Die folgenden Webseiten der Prüfungsordnungen haben sehr lange Ladezeiten!
##  Ein-Fach-Bachelorstudiengänge 
  * [Archäologische Wissenschaften](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/1-fach-bachelor/#archaeologie)
  * [Islamisch-Religiöse Studien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/1-fach-bachelor/#irs)
  * [Literatur und Buch](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/1-fach-bachelor/#literatur-buch)
  * [Psychologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#psychologie)
  * [Soziologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/1-fach-bachelor/#soziologie)
  * [Sportwissenschaft (berufsbegleitend)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/1-fach-bachelor/#sportwissenschaft)


##  Zwei-Fach-Bachelorstudiengänge 
  * [Archäologische Wissenschaften](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#archaeologie)
  * [Buchwissenschaft](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#buchwissenschaft)
  * [Computerlinguistik (ab WS 2022/23; vormals Linguistische Informatik)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#computerlinguistik)
  * [Digitale Geistes- und Sozialwissenschaften (vormals Informatik)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#digitale-geistes-und-sozialwissenschaften)
  * [English and American Studies](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#english-and-american-studies)
  * [Frankoromanistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#frankoromanistik)
  * [Germanistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#germanistik)
  * [Geschichte](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#geschichte)
  * [Griechische Philologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#griechische-philosophie)
  * [Iberoromanistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#iberoromanistik)
  * [Indogermanistik und Indoiranistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#indogermanistik)
  * [Informatik (jetzt Digitale Geistes- und Sozialwissenschaften)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#informatik)
  * [Islamisch-Religiöse Studien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#irs)
  * [Italoromanistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#italoromanistik)
  * [Japanologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#japanologie)
  * [Kulturgeographie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#kulturgeographie)
  * [Kulturgeschichte des Christentums](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#kulturgeschichte-des-christentums)
  * [Kunstgeschichte](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#kunstgeschichte)
  * [Lateinische Philologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#lateinische-philologie)
  * [Linguistische Informatik (ab WS 2022/23: Computerlinguistik)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#linguistische-informatik)
  * [Mittel- und Neulatein](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#mittellatein)
  * [Nordische Philologie (ab WS 2019/20: Skandinavistik)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#nordische-philologie)
  * [Öffentliches Recht](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#oeffentliches-recht)
  * [Ökonomie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#oekonomie)
  * [Orientalistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#orientalistik)
  * [Pädagogik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#paedagogik)
  * [Philosophie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#philosophie)
  * [Politikwissenschaft](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#politikwissenschaft)
  * [Sinologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#sinologie)
  * [Skandinavistik (ab WS 2019/20; vormals Nordische Philologie)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#skandinavistik)
  * [Soziologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#soziologie)
  * [Theater- und Medienwissenschaft](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/2-fach-bachelor/#theater-und-medienwissenschaft)


##  Modul- und Zusatzstudien 
  * [Digital Humanities](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/modulstudien-und-zusatzstudien/#digital-humanties)
  * [Kulturraum Italien – Kunst, Literatur und Sprache](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/modulstudien-und-zusatzstudien/#kunstraum-italien)
  * [Studium Philosophicum](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/modulstudien-und-zusatzstudien/#studium-philosophicum)
  * [Geowissenschaften (Zusatzstudien)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/modulstudien-und-zusatzstudien/#geowissenschaften-zusatz)
  * [Allgemeine und fachbezogene Bildung in der digitalen Welt (Zusatzstudien)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/modulstudien-und-zusatzstudien/#digitale-welt)
  * [Zusatzstudien Lehramt International (Zusatzstudien)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/modulstudien-und-zusatzstudien/#lehramt-international-zusatz)


##  Masterstudiengänge 
  * [The Americas/Las Américas](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#las-americas)
  * [Antike Sprachen und Kulturen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#antike-sprachen)
  * [Arabistik, Islamwissenschaft, Semitistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#arabistik)
  * [Archäologische Wissenschaften](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#archaeologie)
  * [Buchwissenschaft (ab WS 2023/24 Schriftmedienkultur und digitale Transformation)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#buchwissenschaft)
  * [Chinese Studies with an optional focus](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#chinese-studies)
  * [Development Economics and International Studies](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#deis)
  * [Digital Humanities](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#digital-humanities)
  * [Digitale Japanstudien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#digitale-japanstudien)
  * [English Studies](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#english-studies)
  * [Erziehungswissenschaftlich-Empirische Bildungsforschung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#eeb)
  * [Germanistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#germanistik)
  * [Geschichte](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#geschichte)
  * [Imperien und Transkontinentale Räume](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#imperien)
  * [Islamisch-Religiöse Studien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#irs)
  * [Komparatistische Romanistik (siehe auch neu Romanistik)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#komparatistische-romanistik)
  * [Kunstgeschichte](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#kunstgeschichte)
  * [Kunstvermittlung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#kunstvermittlung)
  * [Learning Design – Digitale Transformation in der Bildung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#learning-design)
  * [Linguistik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#linguistik)
  * [Literaturstudien – intermedial und interkulturell (ab WS 2024/25 Literaturstudien – medial und transkulturell)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#literaturstudien)
  * [Literaturstudien – medial und transkulturell (ab WS 2024/25; vormals Literaturstudien – intermedial und interkulturell)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#literaturstudien)
  * [Medienwissenschaft](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#medienwissenschaft)
  * [Mittelalter- und Frühe Neuzeit](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#mittelalter)
  * [Mittellatein und Neulatein](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#mittellatein)
  * [Nahoststudien](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#nahoststudien)
  * [North American Studies: Culture and Literature](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#north-american-studies)
  * [Pädagogik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#paedagogik)
  * [Philosophie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#philosophie)
  * [Politikwissenschaft](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#politikwissenschaft)
  * [Populär- und Medienkultur Japans](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#popkultur-japan)
  * [Romanistik (siehe auch ehem. Komparatistische Romanistik)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#romanistik)
  * [Sinologie mit fachspezifischer Ausrichtung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#sinologie)
  * [Soziologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#soziologie)
  * [Schriftmedienkultur und digitale Transformation (ehemals Buchwissenschaft)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#schriftmedienkultur-digitale-transformation)
  * [Theater – Forschung – Vermittlung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#theater-medien-vermittlung)
  * [Theater- und Medienwissenschaft](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#theater-medienwissenschaft)
  * [Theaterpädagogik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#theaterpaedagogik)


##  Masterstudiengänge mit eigenständiger Prüfungsordnung 
  * [Gerontologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#gerontologie)
  * [Lexicography](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#lexicography)
  * [Physical Activity and Health](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#physical-activity)
  * [Psychologie](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#psychologie)
  * [Psychologie mit Schwerpunkt Klinische Psychologie und Psychotherapie (M.Sc.) (ab WiSe 2022/2023)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/masterstudiengaenge/#klinische-psychologie)


##  Weiterbildungsmasterstudiengänge 
  * [Human rights](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/#human-rights)
  * [Multimedia Didaktik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/#mmd)
  * [Organisations- und Personalentwicklung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/#oepe)


##  Elitemasterstudiengänge 
  * [Ethik der Textkulturen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/#ethik-der-textkulturen)
  * [Standards of Decision-Making Across Cultures](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/#SDAC)


##  Bakkalaureus-, Magister- und Diplomstudiengänge 
  * [Bakkalaureusstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/#bakkalaureus)
  * [Magisterstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/#magister)
  * [Diplomstudiengänge](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/weiterbildung-elite-weitere/#diplom)


##  Studiengänge im Fachbereich Theologie 
#### 2-Fach-Bachelorstudiengänge
  * [Kulturgeschichte des Christentums](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/philosophische-fakultaet-und-fachbereich-theologie/#kulturgeschichte-christentum)
  * [Religion](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/philosophische-fakultaet-und-fachbereich-theologie/#religion)


#### Magisterstudiengang
  * [Ev. Theologie (Abschluss Magister Theologiae – 1. kirchliche Prüfung)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/philosophische-fakultaet-und-fachbereich-theologie/#ev-theologie)


#### Masterstudiengänge
  * [Christliche Medienkommunikation](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/philosophische-fakultaet-und-fachbereich-theologie/#christliche-medienkommunikation)
  * [Medien-Ethik-Religion](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/philosophische-fakultaet-und-fachbereich-theologie/#medien-ethik-religion)


#### Kirchliches Examen (Pfarramt)
  * [Theologische Aufnahmeprüfung (1. und 2. Kirchliches Examen)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/philosophische-fakultaet-und-fachbereich-theologie/#theol-aufnahmepruefung)


#### Diplomstudiengang Evangelische Theologie
  * [Diplomprüfungsordnung und Zwischenprüfungsordnung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/phil/philosophische-fakultaet-und-fachbereich-theologie/#collapse_13)


##  Lehramtsstudiengänge 
  * [Erste und zweite Staatsprüfung](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/)
  * [Fachstudien- und Prüfungsordnungen der Fächer (FAPO)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/lehramtsfaecher/)
  * [LAPO (Modulprüfungen im Rahmen der Ersten Lehramtsprüfung sowie den lehramtsbezogenen Masterstudiengang Gymnasium)](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/#module)
  * [Bachelor- und Masterstudiengänge Berufspädagogik](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/berufspaedagogik-und-zusatzstudien/)
  * [Zusatzfächer](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/lehramt/berufspaedagogik-und-zusatzstudien/)


### Weitere Regelungen
  * [Sprachprüfungen](https://www.fau.de/fau/rechtsgrundlagen/pruefungsordnungen/sprachpruefungen/ "Sprachprüfungen")
  * [Promotions- und Habilitationsordnung](https://www.fau.de/fau/rechtsgrundlagen/promotions-und-habilitationsordnungen/ "Promotions- und Habilitationsordnung")




🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/veranstaltungen/veranstaltungen-der-rechts-und-wirtschaftswissenschaftlichen-fakultaet
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Rechts- und Wirtschaftswissenschaftliche Fakultät
# Rechts- und Wirtschaftswissenschaftliche Fakultät
Überblick über die öffentlichen Termine der Fakultät
Termine für weitere Zielgruppen finden Sie im [Veranstaltungskalender der Fakultät](https://www.rw.fau.de/event/).
### Termine des Fachbereichs Rechtswissenschaften
[www.jura.fau.de](https://www.jura.fau.de)
  * 23
Juli
16:15 – 17:45
[Vortrag Menschenwürde und Metaphysik](https://www.fau.de/termine/vortrag-menschenwuerde-und-metaphysik/)
JDC 1.281, Schillerstr. 1, 91054 Erlangen


[Zum Kalender hinzufügen](https://www.fau.de?ical-plugin=rrze-calendar&action=export&filename=www-fau-de-veranstaltungen-veranstaltungen-der-rechts-und-wirtschaftswissenschaftlichen-fakultaet&ids=18091976)
### Termine des Fachbereichs Wirtschafts- und Sozialwissenschaften
[www.wiso.fau.de](https://www.wiso.fau.de)
  * 15
Juli
15:00 – 17:00
[Promotionsfeier an der FAU WiSo](https://www.fau.de/termine/promotionsfeier-an-der-fau-wiso/)
FAU WiSo, Hörsaal H5, Lange Gasse 20, Nürnberg
  * 22
Juli
17:00 – 18:00
[Infoveranstaltung für den berufsbegleitenden Masterstudiengang „Sustainability Management“ (MBA)](https://www.fau.de/termine/infoveranstaltung-fuer-den-berufsbegleitenden-masterstudiengang-sustainability-management-mba-4/)
Online
  * 22
Juli
17:00 – 18:00
[Online-Infoveranstaltung zum berufsbegleitenden MBA-Studiengang „Sustainability Management“](https://www.fau.de/termine/online-infoveranstaltung-zum-berufsbegleitenden-mba-studiengang-sustainability-management/)
Online
  * 23
Juli
[Ringvorlesung Nachhaltigkeit: Klausur](https://www.fau.de/termine/ringvorlesung-nachhaltigkeit-klausur/)
  * 25
Juli
[Vorlesungsende Sommersemester 2025](https://www.fau.de/termine/vorlesungsende-sommersemester-2025-2/)
  * 31
Juli
10:00 – 16:00
[KoWinChi Workshop: Finanzierung wissenschaftlicher Kooperation zwischen Deutschland und China](https://www.fau.de/termine/kowinchi-workshop-finanzierung-wissenschaftlicher-kooperation-zwischen-deutschland-und-china/)
KoWinChi, Erlangen
  * 11
Sep.
[Workshop on Heterogeneous Macro Expectations – New Evidence and Theory](https://www.fau.de/termine/workshop-on-heterogeneous-macro-expectations-new-evidence-and-theory/)
FAU WiSo
  * 12
Sep.
[Workshop on Heterogeneous Macro Expectations – New Evidence and Theory](https://www.fau.de/termine/workshop-on-heterogeneous-macro-expectations-new-evidence-and-theory-2/)
FAU WiSo
  * 24
Sep.
24. September 2025 – 26. September 2025
[FAU Studieninfotage](https://www.fau.de/termine/fau-studieninfotage/)
  * 25
Sep.
24. September 2025 – 26. September 2025
[FAU Studieninfotage](https://www.fau.de/termine/fau-studieninfotage/)
  * 30
Sep.
[Bewerbungsschluss Bachelorstudiengänge zum Wintersemester 2025/2026](https://www.fau.de/termine/bewerbungsschluss-bachelorstudiengaenge-zum-wintersemester-2025-2026/)
  * 30
Sep.
[Ende des Sommersemesters 2025](https://www.fau.de/termine/ende-des-sommersemesters-2025/)
  * 30
Sep.
[Workshop “Business, Human Rights and Environment in the Andean Countries”](https://www.fau.de/termine/workshop-business-human-rights-and-environment-in-the-andean-countries/)
  * 08
Okt.
10:00 – 16:00
[KoWinChi Workshop: A Collaborative Exchange Between German and Chinese Researchers](https://www.fau.de/termine/kowinchi-workshop-a-collaborative-exchange-between-german-and-chinese-researchers/)
Erlangen
  * 10
Okt.
09:00 – 17:00
[Seminarprogramm der Frauenbeauftragten: Disputationstraining für Nachwuchswissenschaftlerinnen](https://www.fau.de/termine/seminarprogramm-der-frauenbeauftragten-disputationstraining-fuer-nachwuchswissenschaftlerinnen-3/)
Online
  * 13
Okt.
[Erstsemesterbegrüßung und Studienstart an der FAU WiSo](https://www.fau.de/termine/erstsemesterbegruessung-und-studienstart-an-der-fau-wiso/)
FAU WiSo, Lange Gasse 20, Nürnberg
  * 17
Okt.
09:00 – 17:00
[Seminarprogramm der Frauenbeauftragten: Disputationstraining für Nachwuchswissenschaftlerinnen](https://www.fau.de/termine/seminarprogramm-der-frauenbeauftragten-disputationstraining-fuer-nachwuchswissenschaftlerinnen-4/)
Online
  * 25
Okt.
[Lange Nacht der Wissenschaften an der FAU WiSo](https://www.fau.de/termine/lange-nacht-der-wissenschaften-an-der-fau-wiso/)
FAU WiSo, Lange Gasse 20, Nürnberg
  * 27
Okt.
09:30 – 17:00
[Seminarprogramm der Frauenbeauftragten: Strategisch Netzwerken für Nachwuchswissenschaftlerinnen](https://www.fau.de/termine/seminarprogramm-der-frauenbeauftragten-strategisch-netzwerken-fuer-nachwuchswissenschaftlerinnen/)
Lange Gasse 20, Raum 4.154
  * 28
Okt.
10:00 – 16:00
[KoWinChi Workshop: Chinas Wissenschaftssystem: Universitäten](https://www.fau.de/termine/kowinchi-workshop-chinas-wissenschaftssystem-universitaeten/)
Erlangen
  * 11
Nov.
14:00 – 15:00
[Seminarprogramm der Frauenbeauftragten: Forschungsförderung in der Wissenschaft – Themennachmittag](https://www.fau.de/termine/seminarprogramm-der-frauenbeauftragten-forschungsfoerderung-in-der-wissenschaft-themennachmittag/)
Online
  * 14
Nov.
10:00 – 16:00
[KoWinChi Workshop: Wissenschaftsfreiheit und -ethik in Deutschland und China](https://www.fau.de/termine/kowinchi-workshop-wissenschaftsfreiheit-und-ethik-in-deutschland-und-china-2/)
Erlangen
  * 20
Nov.
09:00 – 16:00
[Seminarprogramm der Frauenbeauftragten: Career Development](https://www.fau.de/termine/seminarprogramm-der-frauenbeauftragten-career-development/)
Online
  * 12
Dez.
10:00 – 16:00
[KoWinChi Workshop: Wissenschaft und (internationale) Politik](https://www.fau.de/termine/kowinchi-workshop-wissenschaft-und-internationale-politik/)
Erlangen


[Zum Kalender hinzufügen](https://www.fau.de?ical-plugin=rrze-calendar&action=export&filename=www-fau-de-veranstaltungen-veranstaltungen-der-rechts-und-wirtschaftswissenschaftlichen-fakultaet&ids=18092050,18092051,18092052,18092053,18092054,18092055,18092056,18092057,18092058,18092059,18092060,18092061,18092062,18092063,18092064,18092065,18092066,18092067,18092068,18092069,18092070,18092071,18092072)


...

🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/graduiertenzentrum/strukturierte-promotionsprogramme/rechts-wirtschafts-und-sozialwissenschaften
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Rechts-, Wirtschafts- und Sozialwissenschaften
# Rechts-, Wirtschafts- und Sozialwissenschaften
Strukturierte Promotionsprogramme Rechts-, Wirtschafts- und Sozialwissenschaften
##  Business and Human Rights: Governance Challenges in a Complex World (IDP B&HR_Governance) 
### Keywords
Menschenrechte, transnationale Wirtschaftsbeziehungen, unternehmerische Verantwortung, Arbeitsmigration, Nachhaltigkeit, digitale Transformation
### Doktortitel
Dr. jur., Dr. rer. pol., Dr. phil.
### Programmsprache
Englisch
### Programmdauer
4 Jahre
### Bewerbung
Weitere Informationen zu laufenden Bewerbungen können Sie hier finden: <https://www.business-humanrights.fau.eu/>
### Besonderheiten
Das Internationale Promotionskolleg „ Business and Human Rights: Governance Challenges in a Complex World“ wird vom Elitenetzwerk Bayern gefördert und umfasst 12 Promovenden, die an der FAU im Umfang von 2/3 einer E13/A13-Stelle angestellt sind. Das Programm sieht ein umfangreiches thematisches und methodisches Curriculum sowie eine intensive Betreuung durch Hochschullehrer/-innen der FAU und der zusätzlich beteiligten Universitäten Bayreuth und Würzburg vor.
### FAU-Ansprechpartner und Informationen zum Programm
[Prof. Dr. Markus Krajewski](https://faudir.fau.de/public/person/00b4c0a78c)  
Hyun Jung Lee, M.A.
[Informationen zum Graduiertenkolleg](https://www.rph1.rw.fau.de/forschung/international-doctorate-programme-business-and-human-rights/ "Informationen zum Graduiertenkolleg")
##  Cyberkriminalität und Forensische Informatik (DFG GRK 2475) 
### Keywords
forensische Informatik, Cyberkriminalität, Strafrecht, Strafprozessrecht, digitale Spuren
### Doktortitel
Dr.-Ing., Dr. jur.
### Programmsprache
Deutsch und Englisch
### Programmdauer
Die Förderung ist zunächst für 4,5 Jahre, Promotionsförderung für maximal 3 Jahre pro Promovend.
### Bewerbung
Über die Homepage des GRK ([cybercrime.fau.de](http://www.cybercrime.fau.de "cybercrime.fau.de")).
### Besonderheiten
Die Besonderheit des Programms ist, dass es die Informatik mit den Rechtswissenschaften verbindet und die intensive interdisziplinäre Betreuung. Geförderte Promovierende erhalten eine 100% (Informatik) bzw. 75% (Rechtswissenschaften) Stelle für maximal 3 Jahre. Es gibt für anderweitig geförderte Promovierende bei thematischer Nähe zum GRK die Möglichkeit der Assoziierung (Teilnahme an den Veranstaltungen des GRK, Partizipation an der Infrastruktur, etc.).
### FAU-Ansprechpartner und Informationen zum Programm
[Felix Freiling](https://faudir.fau.de/public/person/41053516a2) (Sprecher) und [Hans Kudlich](https://faudir.fau.de/public/person/8ba915f22e) (stellvertretender Sprecher)
[Informationen zum Graduiertenkolleg](http://www.cybercrime.fau.de "Informationen zum Graduiertenkolleg")
##  Evidence-Based-Economics (ENB) 
### Keywords
Evidenzbasierte Wirtschaftsforschung, Evaluierung von politischen Maßnahmen, Angewandte Wirtschaftsforschung, Labor- und Feldversuche, Mikroökonomie
### Doktortitel
Dr. rer. pol.
### Programmsprache
Englisch
### Programmdauer
3 bis 4 Jahre
### Bewerbung
Über ein von der Munich Graduate School of Ecomomics verwaltetes [Online-Bewerbungs-Tool](https://www.portal.graduatecenter.lmu.de/gc-application/de/home). Detaillierte Bewerbungsvoraussetzungen finden Sie auf der [Homepage.](http://www.ebe.econ.uni-muenchen.de/admission/application/index.html)
### Besonderheiten
Kooperation (EBE ist eine vom Elitenetzwerk Bayern großzügig geförderte Gemeinschaftsunternehmung der Universitäten München, Erlangen-Nürnberg und Regensburg), Methodologischer Fokus, [Strukturierter Studienplan](http://www.ebe.econ.uni-muenchen.de/doctoral-program/curriculum/index.html), Networking-Veranstaltungen, feste Anstellung: 75% TV-L E 13, Reisekostenunterstützung für Konferenzen, Austauschbesuche und Summer schools, Anschubfinanzierung, Unterstützung für Sprachkurse, Gleichstellungsmittel, Soft-Skill-Kurse
### Informationen zum Programm
[Informationen zum Programm](http://www.ebe.econ.uni-muenchen.de/about-ebe/index.html)
##  GradAB am Institut für Arbeitsmarkt- und Berufsforschung (IAB) 
### Keywords
Arbeitsmarktforschung, Wirtschaft, Soziologie, Sozialwissenschaften, Politikberatung, Ökonometrie
### Doktortitel
Dr. rer. pol.
### Programmsprache
Englisch
### Programmdauer
3 Jahre (+ 1 Jahr unter bestimmten Bedingungen)
### Bewerbung
An [Dr. Sandra Huber](http://www.iab.de/en/ueberblick/graduiertenprogramm/bewerbung.aspx)
### Besonderheiten
Stipendium über 1600 €/Monat; 150 €/Monat für Konferenzen, Summer Schools, Forschungsaufenthalte, jeder Doktorand wird von einem Mentor des IAB unterstützt
### FAU-Ansprechpartner und Informationen zum Programm
[Prof. Dr. Claus Schnabel](https://faudir.fau.de/public/person/a41c2ec288)
[Prof. Dr. Martin Abraham](https://faudir.fau.de/public/person/a8285dbd3d)
[Informationen zum Programm](https://iab.de/en/gradab/)
##  Graduate Programme in Sociology 
### Keywords
Soziologische Theorie, Arbeits- und Organisationssoziologie, Vergleichende Gesellschaftsanalyse, Bildung und Lebenslauf, Kultur und Kommunikation, Methoden der empirischen Sozialforschung
### Doktortitel
Dr. phil.
### Programmsprache
Deutsch
### Programmdauer
3 bis 4 Jahre
### Bewerbung
Direkte Ansprache der für die [einzelnen Bereiche zuständigen Professorinnen und Professoren](https://www.soziologie.phil.fau.de/institut/team/)
Soziologische Theorie: Prof. Dr. Silke Steets
Arbeits- und Organisationssoziologie: Prof. Dr. Rainer Trinczek
Vergleichende Gesellschaftsanalyse: Prof. Dr. Ingrid Artus
Bildung und Lebenslauf: Prof. Dr. Renate Liebold
Kultur und Kommunikation: Prof. Dr. Silke Steets
Methoden der empirischen Sozialforschung: Prof. Dr. Nicole Saam, Prof. Dr. Renate Liebold
### Besonderheiten
Möglichkeit der Teilnahme an thematisch breit gefächerten Lehrveranstaltungen und Oberseminaren
### FAU-Ansprechpartner und Informationen zum Programm
[Prof. Dr. Ingrid Artus](https://faudir.fau.de/public/person/84d6829ba3)
[Informationen zum Programm](https://www.soziologie.phil.fau.de/studium/studiengaenge/graduate-studies/)
##  Graduiertenschule des Fachbereichs Wirtschaftswissenschaften 
### Keywords
Arbeit im Wandel, Customer Insights, Steuern und Steuerpolitik, Versicherungen und Risiko, Digitalisierung der Wirtschaft, Energiemärkte und Energiesystemanalyse, Gesundheit
### Doktortitel
Dr. rer. pol.
### Programmdauer
3 bis 5 Jahre
### Bewerbung
Bitte suchen Sie sich zunächst einen Betreuer/eine Betreuerin innerhalb der [FAU/Institute](https://www.fau.de/graduiertenzentrum/promotion/promotionsmodule/)
### FAU-Ansprechpartner und Informationen zum Programm
Bitte suchen Sie sich einen Professor innerhalb der [Institute](https://www.wiso.rw.fau.de/forschung/forschungsprofil/institute/)
[Informationen zur Graduiertenschule](https://www.rw.fau.de/forschung/graduiertenschule/)
##  Menschenrechte und Ethik in der Medizin für Ältere 
### Keywords
Menschenrechte, Ethik, Medizin, Ältere Menschen, Autonomie, Medizinrecht
### Doktortitel
Dr. phil., Dr. med., Dr. jur.
### Programmsprache
Deutsch
### Programmdauer
3,5 Jahre
### Bewerbung
Aktuelle Bewerbungsfrist bis zum 15. September 2021. Informationen über etwaige neue Ausschreibungen über die [Homepage des Kollegs.](https://www.grk.menschenrechte-und-ethik.med.fau.de/)
### Besonderheiten
Keine strikte Zeitbegrenzung, finanzielle Förderung durch die Josef und Luise Kraft-Stiftung (München), Mentoring mit Tandem-Betreuung durch jeweils zwei Personen (interdisziplinär), Kooperation mit diversen Institutionen: Centre for Human Rights Erlangen-Nürnberg (CHREN), Klinisches Ethikkomitee am UK Erlangen (KEK) etc.
### FAU-Ansprechpartner und Informationen zum Programm
[Prof. Dr. med. Andreas Frewer, M.A.](https://faudir.fau.de/public/person/83229933e7)
[Prof. Dr. Dr. h.c. Heiner Bielefeldt](https://faudir.fau.de/public/person/6f2b649392)
[Dipl.-Pol. Sabine Klotz](https://faudir.fau.de/internal/person/c255f3f68a)
[Informationen zum Programm](https://www.grk.menschenrechte-und-ethik.med.fau.de/)
##  Promotionsprogramm Bewegung und Gesundheit des ISS (Institut für Sportwissenschaft und Sport) 
### Keywords
Bewegung, Training, Gesundheit, Bewegungstherapie, public health
### Doktortitel
Dr. phil.
### Programmsprache
Deutsch
### Programmdauer
2 Jahre
### Bewerbung
[Ansprechpartner](https://www.sport.fau.de/studium/promotionsprogramm/kontakt/) sind die Koordinatoren und Vorstandssprecher des Promotionsprogramms
### FAU-Ansprechpartner und Informationen zum Programm
[Dr. Jana Semrau](https://faudir.fau.de/public/person/44db153fd0)
[Informationen zum Programm](https://www.sport.fau.de/studium/promotionsprogramm/)
##  Promotionsprogramm Gerontologie des ICA (Interdisziplinäres Centrum für Alternsforschung) 
### Keywords
Altern und Technologie, Ernährung im Alter, Physische Frailty, Anpassung im Alter, Demenz, Depression im Alter
### Doktortitel
Dr. phil. und Dr. rer. biol. hum.
### Programmsprache
Deutsch und Englisch
### Programmdauer
2 bis 3 Jahre, je nach angestrebtem Doktortitel
### Bewerbung
Aufnahme über Erstbetreuer der Doktorarbeit, wenn dieser zugleich Mitglied im strukturierten Programm bzw. im Interdisziplinären Centrum für Altersforschung ist. Direkte Bewerbung ist nicht möglich.
### Besonderheiten
Das Promotionsprogramm vergibt selbst keine Stipendien, unterstützt Doktorandinnen und Doktoranden aber aktiv in der Antragstellung.
### FAU-Ansprechpartner und Informationen zum Programm
[Prof. Dr. Frieder R. Lang](https://faudir.fau.de/public/person/22e79d8950)
[Informationen zum Programm](http://www.geronto.fau.de/lehre/promotionsprogramm/)
##  SeReCo: Semantics, Reasoning and Coordination Technologies 
### Keywords
Web of Data, Linked Data, Semantic Web, Multi-Agent Systems, Intelligent Agents, Web of Things, Sensor Networks
### Doktortitel
Dr. rer. pol.
### Programmsprache
Englisch
### Programmdauer
4 Jahre
### Bewerbung
Per E-Mail an Prof. Dr. Andreas Harth (andreas.harth@fau.de)
### Besonderheiten
  1.      * Internationale Kooperation der FAU mit École des Mines de Saint-Étienne, Universität Jean Monnet Saint-Étienne, Universität St. Gallen und Karlsruher Institut für Technologie
     * Förderung durch die Deutsch-Französische Hochschule
     * Jährliche Workshops für Doktoranden
     * Unterstützung für Konferenzen, Tagungen, Kongressen, Forschungsaufenthalte im Ausland möglich


### FAU-Ansprechpartner
[Prof. Dr. Andreas Harth](https://faudir.fau.de/public/person/1dff93c83c)
[Informationen zum Programm](https://www.ti.rw.fau.de/doctoral-college-semantics-reasoning-and-coordination-technologies-sereco/)


🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/termine/bachelor-kulturgeographie-physische-geographie-und-lehramt-geographie-einfuehrungsveranstaltung
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Bachelor Kulturgeographie, Physische Geographie und Lehramt Geographie: Einführungsveranstaltung
# Bachelor Kulturgeographie, Physische Geographie und Lehramt Geographie: Einführungsveranstaltung
Datum: 14. Oktober 2024Zeit: 10:15 – 11:45Ort: Hörsaal C, Kochstr. 4, Erlangen
Einführungsveranstaltung für alle Erstsemesterstudierenden der Bachelorstudiengänge Physische Geographie und Kulturgeographie sowie für die Lehramtsstudiengänge Geographie
[Zum Kalender hinzufügen](https://www.fau.de?ical-plugin=rrze-calendar&action=export&filename=www-fau-de-termine-bachelor-kulturgeographie-physische-geographie-und-lehramt-geographie-einfuehrungsveranstaltung&ids=18092883)
## Details Datum:
    14. Oktober 2024 

Zeit:
    10:15 – 11:45 

Ort:
    
Hörsaal C, Kochstr. 4, Erlangen 

Veranstaltungskategorien:
    [Naturwissenschaftliche Fakultät](https://www.fau.de/calendars/natfak/)


🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/education/studienangebot
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Studienangebot
# Studienangebot
## Viele Wege – ein Ziel: Bildung
Wann immer Sie sich für ein Studium entscheiden – direkt nach dem Abitur oder später – als Volluniversität bietet Ihnen die FAU das gesamte Spektrum an Studienfächern und eine Vielzahl an Studiengängen.
### Einzigartiges Studienangebot
Unser Studienangebot aus über 260 Studiengängen ist in seiner Vielfalt und interdisziplinären Vernetzung deutschlandweit einzigartig. Unsere Plattform FAU MeinStudium unterstützt als virtueller Begleiter auf dem Weg zum Studium an unserer Universität.
[Übersicht und ausführliche Infos zu den FAU-Studiengängen](https://www.fau.de/education/studienangebot/alle-studiengaenge/)
  

### Infoveranstaltungen für Studieninteressierte
[Veranstaltungen für Studieninteressierte, Schülerinnen und Schüler](https://meinstudium.fau.de/termine-fuer-studieninteressierte/)
### Starten Sie Ihr Studium an der FAU!
[Infos zum Studienstart](https://www.fau.de/education/studienorganisation/studienstart/)
  

Studienabschlüsse im Überblick
Unsere Staatsexamens- und Bachelorstudiengänge sind der Einstieg für alle, die noch kein Studium abgeschlossen haben. Im Masterstudium werden das Fachwissen nach einem abgeschlossenen Studium ausgebaut und die wissenschaftlichen Inhalte vertieft. Zudem bieten wir Lehramtsstudiengänge für Grundschule, Mittelschule, Realschule, Gymnasium und Berufsschule an.
  * [](https://www.fau.de/education/studienangebot/bachelorstudiengaenge/)
[Bachelorstudiengänge](https://www.fau.de/education/studienangebot/bachelorstudiengaenge/)
  * [](https://www.fau.de/education/studienangebot/masterstudiengaenge/)
[Masterstudiengänge](https://www.fau.de/education/studienangebot/masterstudiengaenge/)
  * [](https://www.fau.de/education/studienangebot/lehramtsstudium/)
[Lehramtsstudiengänge](https://www.fau.de/education/studienangebot/lehramtsstudium/)
  * [](https://www.fau.de/education/studienangebot/staatsexamen-ausser-lehramt/)
[Staatsexamen (außer Lehramt)](https://www.fau.de/education/studienangebot/staatsexamen-ausser-lehramt/)


### Besondere Studienformen
Wir haben zusätzliche Studienformen entwickelt, um etwa das Studieren in Teilzeit zu ermöglichen. Unser duales Bachelorverbundstudium in Kooperation mit der IHK Nürnberg für Mittelfranken und der HWK für Mittelfranken ist dabei deutschlandweit bislang einmalig.
  * [](https://dual.fau.de)
[Duales Studium / Bachelorverbundstudium](https://dual.fau.de)
  * [](https://www.fau.de/education/studienangebot/teilzeitstudium/)
[Teilzeitstudium](https://www.fau.de/education/studienangebot/teilzeitstudium/)
  * [](https://www.fau.de/education/studienangebot/berufsbegleitendes-studium/)
[Berufsbegleitendes Studium](https://www.fau.de/education/studienangebot/berufsbegleitendes-studium/)
  * [](https://www.fau.de/education/studienangebot/senioren-und-gaststudium/)
[Gaststudium](https://www.fau.de/education/studienangebot/senioren-und-gaststudium/)
  * [](https://www.fau.de/education/studienangebot/zweitstudium/)
[Zweitstudium](https://www.fau.de/education/studienangebot/zweitstudium/)
  * [](https://www.fau.de/education/schule-und-uni/schuelerinnen-und-schueler/fruehstudium/)
[Frühstudium](https://www.fau.de/education/schule-und-uni/schuelerinnen-und-schueler/fruehstudium/)
  * [](https://www.fau.de/education/studienangebot/schnupperstudium/)
[Schnupperstudium](https://www.fau.de/education/studienangebot/schnupperstudium/)
  * [](https://www.fau.de/education/studienangebot/modulstudien/)
[Modulstudien](https://www.fau.de/education/studienangebot/modulstudien/)


### Spezielle Studienmöglichkeiten und studienbegleitende Angebote
Um auch auf den globalen Arbeitsmarkt bestens vorbereitet zu sein, bieten wir beispielsweise internationale Studiengänge an. In allen Studiengängen der FAU können studienbegleitend Fremdsprachenkenntnisse bei uns erworben werden.
  * [](https://www.fau.de/education/studienangebot/internationale-studiengaenge/doppelabschluss-studiengaenge/)
[Studiengänge mit Doppelabschluss](https://www.fau.de/education/studienangebot/internationale-studiengaenge/doppelabschluss-studiengaenge/)
  * [](https://www.fau.de/education/studienangebot/elitestudiengaenge/)
[Elitestudiengänge](https://www.fau.de/education/studienangebot/elitestudiengaenge/)
  * [](https://www.fau.de/education/studienangebot/internationale-studiengaenge/)
[Internationale Studiengänge](https://www.fau.de/education/studienangebot/internationale-studiengaenge/)
  * [](https://www.fau.de/education/studienangebot/promotionsprogramme/)
[Promotion an der FAU](https://www.fau.de/education/studienangebot/promotionsprogramme/)
  * [](https://www.fau.de/education/studienangebot/weiterbildungsstudiengaenge/)
[Weiterbildung und Weiterbildungsstudiengänge](https://www.fau.de/education/studienangebot/weiterbildungsstudiengaenge/)
  * [](https://www.fau.de/education/studienangebot/studienbegleitende-fremdsprachenausbildung/)
[Studienbegleitende Fremdsprachenausbildung](https://www.fau.de/education/studienangebot/studienbegleitende-fremdsprachenausbildung/)
  * [](https://www.fau.de/education/studienangebot/virtuelle-hochschule-bayern/)
[Virtuelle Hochschule Bayern (vhb)](https://www.fau.de/education/studienangebot/virtuelle-hochschule-bayern/)
  * [](https://www.rrze.fau.de/ausbildung-schulung/schulungszentrum/)
[IT-Schulungen am RRZE](https://www.rrze.fau.de/ausbildung-schulung/schulungszentrum/)




🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.eu/2019/07/news/a-birthday-cake-for-the-school-of-business-economics-and-society/
Simulate organization breadcrumb open
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.eu/)
A birthday cake for the School of Business, Economics, and Society
# A birthday cake for the School of Business, Economics, and Society
[](https://www.fau.eu/files/2019/07/20190626_100JahreWiSo-Lange-Tafel_Iannicelli8990.jpg)Image: FAU/Giulia Iannicelli)
It wouldn’t be a birthday party without cake!
July 10, 2019
FAU gave the School of Business, Economics, and Society a birthday cake, which was three metres long and decorated in the School’s red colour. The cake was cut on 26 June 2019 on the long table set up in the courtyard of the School of Business, Economics, and Society by the President of FAU, Prof. Dr. Joachim Hornegger and Prof. Dr. Veronika Grimm, Speaker of the School and Dean of the Faculty. The Executive Board was also there to enjoy a piece of the cake with staff, students and their families at the party.
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9375.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli8924_9179.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli8925.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, und FAU-Präsident Prof. Dr. Joachim Hornegger schneiden den Kuchen an. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, und FAU-Präsident Prof. Dr. Joachim Hornegger schneiden den Kuchen an. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli8975.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, und FAU-Präsident Prof. Dr. Joachim Hornegger schneiden den Kuchen an. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, und FAU-Präsident Prof. Dr. Joachim Hornegger schneiden den Kuchen an. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli8990.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, und FAU-Präsident Prof. Dr. Joachim Hornegger schneiden den Kuchen an. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, und FAU-Präsident Prof. Dr. Joachim Hornegger schneiden den Kuchen an. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli8991_9163.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli8992_9124.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9093.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, und FAU-Präsident Prof. Dr. Joachim Hornegger verteilen den Kuchen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, und FAU-Präsident Prof. Dr. Joachim Hornegger verteilen den Kuchen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9150.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9190.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9207.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9218.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9231.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9236.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9241.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9245.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, mit ihren Kindern. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, mit ihren Kindern. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9252.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, mit ihrer Familie. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. Im Bild: Prof. Dr. Veronika Grimm, Dekanin der WiSo, mit ihrer Familie. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9258.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9266.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9302.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9308.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9328.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9361.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9368.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli9375.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 
  * ([Enlarge](https://www.doc.zuv.fau.de//M/galerien/2019/0626_100-Jahre-Wiso-Lange-Tafel/20190626_100JahreWiSo-Lange-Tafel_Iannicelli8924_9179.jpg "Der Fachbereich Wirtschafts- und Sozialwissenschaften \(WiSo\) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. \(Bild: FAU/Giulia Iannicelli\)")) 
100 Jahre WiSo: Lange Tafel  
Der Fachbereich Wirtschafts- und Sozialwissenschaften (WiSo) feiert im Jahr 2019 seinen 100. Geburtstag – unter mit einer „Langen Tafel“, die im Innenhof der WiSo zum gemeinsamen Essen, Vernetzen und Verweilen einlädt. Ein Highlight: Die FAU schenkt der WiSo einen Geburtstagskuchen, und alle können mitessen. (Bild: FAU/Giulia Iannicelli) 


  * [Previous](https://www.fau.eu/2019/07/news/a-birthday-cake-for-the-school-of-business-economics-and-society/)
  * [Next](https://www.fau.eu/2019/07/news/a-birthday-cake-for-the-school-of-business-economics-and-society/)


* * *
#### 100 years of the School of Business, Economics, and Society
The School of Business, Economics, and Society (WiSo) at FAU is celebrating its 100th anniversary this year with a series of exciting events such as an exhibition of the School’s history with artworks by artists from the region, a campus festival and the #NUEdialog congress about digitalisation and sustainability. More information is available at [www.wiso100.fau.de](https://www.wiso100.fau.de/).
  * [](https://www.facebook.com/sharer/sharer.php?u=https%3A%2F%2Fwww.fau.eu%2F2019%2F07%2Fnews%2Fa-birthday-cake-for-the-school-of-business-economics-and-society%2F "Share on Facebook")
  * [](https://bsky.app/intent/compose?text=A%20birthday%20cake%20for%20the%20School%20of%20Business%2C%20Economics%2C%20and%20Society%20https%3A%2F%2Fwww.fau.eu%2F2019%2F07%2Fnews%2Fa-birthday-cake-for-the-school-of-business-economics-and-society%2F%20 "Share on Bluesky")
  * [](https://api.whatsapp.com/send?text=https%3A%2F%2Fwww.fau.eu%2F2019%2F07%2Fnews%2Fa-birthday-cake-for-the-school-of-business-economics-and-society%2F%20A%20birthday%20cake%20for%20the%20School%20of%20Business%2C%20Economics%2C%20and%20Society "Share on Whatsapp")
  * [](https://telegram.me/share/url?url=https%3A%2F%2Fwww.fau.eu%2F2019%2F07%2Fnews%2Fa-birthday-cake-for-the-school-of-business-economics-and-society%2F&text=A%20birthday%20cake%20for%20the%20School%20of%20Business%2C%20Economics%2C%20and%20Society "Share on Telegram")
  * [](https://www.xing.com/spi/shares/new?url=https%3A%2F%2Fwww.fau.eu%2F2019%2F07%2Fnews%2Fa-birthday-cake-for-the-school-of-business-economics-and-society%2F "Share on XING")
  * [](https://www.linkedin.com/sharing/share-offsite/?url=https%3A%2F%2Fwww.fau.eu%2F2019%2F07%2Fnews%2Fa-birthday-cake-for-the-school-of-business-economics-and-society%2F "Share on LinkedIn")
  * [](https://www.reddit.com/submit?url=https%3A%2F%2Fwww.fau.eu%2F2019%2F07%2Fnews%2Fa-birthday-cake-for-the-school-of-business-economics-and-society%2F "Share on Reddit")
  * 

## News Archives
News Archives Select Month July 2025  June 2025  May 2025  April 2025  March 2025  February 2025  January 2025  December 2024  November 2024  October 2024  September 2024  August 2024  July 2024  June 2024  May 2024  April 2024  March 2024  February 2024  January 2024  December 2023  November 2023  October 2023  September 2023  August 2023  July 2023  June 2023  May 2023  April 2023  March 2023  February 2023  January 2023  December 2022  November 2022  October 2022  September 2022  August 2022  July 2022  June 2022  May 2022  April 2022  March 2022  February 2022  January 2022  December 2021  November 2021  October 2021  September 2021  August 2021  July 2021  June 2021  May 2021  April 2021  March 2021  February 2021  January 2021  December 2020  November 2020  October 2020  September 2020  August 2020  July 2020  June 2020  May 2020  April 2020  March 2020  February 2020  January 2020  December 2019  November 2019  October 2019  September 2019  August 2019  July 2019  June 2019  May 2019  April 2019  March 2019  February 2019  January 2019  December 2018  November 2018  October 2018  September 2018  August 2018  July 2018  June 2018  May 2018  April 2018  March 2018  February 2018  January 2018  December 2017  November 2017  October 2017  September 2017  August 2017  July 2017  June 2017  May 2017  April 2017  March 2017  February 2017  January 2017  December 2016  November 2016  October 2016  September 2016  August 2016  July 2016  June 2016  May 2016  April 2016  March 2016  February 2016  January 2016  December 2015  November 2015  October 2015  September 2015  August 2015  July 2015  June 2015  May 2015  April 2015  March 2015  February 2015  January 2015  December 2014  November 2014  October 2014  September 2014  August 2014  July 2014  June 2014  May 2014  March 2014  February 2014  January 2014  December 2013  November 2013  October 2013  September 2013  August 2013  July 2013  June 2013  May 2013  April 2013  March 2013  February 2013  January 2013  December 2012  November 2012  October 2012  September 2012  August 2012  July 2012  June 2012  May 2012  April 2012  March 2012  February 2012  January 2012  December 2011  November 2011  October 2011  September 2011  July 2011  June 2011  April 2011  March 2011  December 2010  July 2010  June 2010  April 2010  November 2009  October 2009  June 2009  May 2009 
[More blog entries](https://www.fau.eu/category/news/)


🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
🔶
https://www.fau.de/termine/digitaler-bachelorday-der-fau-wiso-2
Organisationsmenü öffnen
[Friedrich-Alexander-Universität Erlangen-Nürnberg ](https://www.fau.de/)
Digitaler BachelorDay der FAU WiSo
# Digitaler BachelorDay der FAU WiSo
Datum: 1. Juli 2025Zeit: 10:00 – 16:00Ort: Online
Der digitale BachelorDay der FAU WiSo ist die perfekte Möglichkeit, sich von überall auf der Welt via Computer, Tablet oder Smartphone über das Studienangebot und das Campusleben an der WiSo Nürnberg zu informieren.
Es bietet sich für alle interessierten Schülerinnen und Schüler die Möglichkeit, das Studierendenleben in Nürnberg zu entdecken und mehr über das vielfältige Studienangebot im Bachelor zu erfahren. Alle Bachelorstudiengänge der WiSo Nürnberg stellen sich kompakt vor und beantworten Fragen. Zusätzlich gibt die zentrale Studienberatung Informationen zum Zulassungs- und Bewerbungsverfahren.
Das gesamte Programm inklusive Zugangslinks für die Vorträge gibt es hier: <https://www.wiso.rw.fau.de/studium/vor-dem-studium/wiso-bachelorday>
[Zum Kalender hinzufügen](https://www.fau.de?ical-plugin=rrze-calendar&action=export&filename=www-fau-de-termine-digitaler-bachelorday-der-fau-wiso-2&ids=17413663)
## Details Datum:
    1. Juli 2025 

Zeit:
    10:00 – 16:00 

Ort:
    
Online 

Veranstaltungskategorien:
    [Die FAU](https://www.fau.de/calendars/die-fau/)


In [3]:
import time

model = ".elser_model_2"

try:
    client.ml.put_trained_model(model_id=model, input={"field_names": ["text_field"]})
except:
    pass

while True:
    status = client.ml.get_trained_models(model_id=model, include="definition_status")

    if status["trained_model_configs"][0]["fully_defined"]:
        print(model + " is downloaded and ready to be deployed.")
        break
    else:
        print(model + " is downloading or not ready to be deployed.")
    time.sleep(5)

client.ml.start_trained_model_deployment(
    model_id=model, number_of_allocations=1, wait_for="starting"
)

while True:
    status = client.ml.get_trained_models_stats(
        model_id=model,
    )
    if status["trained_model_stats"][0]["deployment_stats"]["state"] == "started":
        print(model + " has been successfully deployed.")
        break
    else:
        print(model + " is currently being deployed.")
    time.sleep(5)

.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deployed.
.elser_model_2 is downloading or not ready to be deploye

ConnectionTimeout: Connection timed out